In [23]:
from flowise import Flowise, PredictionData, IFileUpload
import pandas as pd
import requests
import base64

In [24]:
df = pd.read_csv("data/qa-pair.csv")
df.head()

,Question,True answer,Real answer,Score,Note
0,Chính sách học bổng của trường,Chính sách học bổng\n1. Học bổng CMC Khai phón...,NaN,NaN,NaN
1,Có bao nhiêu phương thức xét tuyển,"Năm 2024, Trường Đại học CMC xét tuyển theo 5 ...",NaN,NaN,NaN
2,Nộp hồ sơ đăng ký xét tuyển,Cách 1: Nộp hồ sơ trực tuyến.\nBước 1: Thí sin...,NaN,NaN,NaN
3,Các khoản phí cần nộp khi nhập học,Khoản học phí tạm thu lần đầu: 11.054.520 đồng...,NaN,NaN,NaN


In [25]:
questions = df.loc[df["Real answer"].isna(), "Question"].values
true_answers = df.loc[df["Real answer"].isna(), "True answer"].values

In [26]:
question, true_answer = questions[0], true_answers[0]
user_query = f"""Question: {question}
True answer: {true_answer}"""
user_query

'Question: Chính sách học bổng của trường\nTrue answer: Chính sách học bổng\n1. Học bổng CMC Khai phóng\nSố lượng dự kiến: 50 suất.\n\nTrị giá học bổng: 100% học phí toàn khóa học (không bao gồm học phí chương trình học tiếng Anh).\n\nĐối tượng và tiêu chí xét học bổng: thí sinh nhập học Trường Đại học CMC năm 2024 thỏa mãn ít nhất một trong các tiêu chí sau:\n\nĐạt giải nhất, nhì, ba trong các kỳ thi học sinh giỏi và kỳ thi Khoa học Kỹ thuật cấp quốc gia, quốc tế trong các năm từ 2021 – 2024 và có môn đoạt giải nằm trong tổ hợp xét tuyển vào ngành tương ứng của Trường Đại học CMC.\nLưu ý: Thí sinh đạt giải môn Tin học được xét học bổng vào ngành Công nghệ Thông tin.\n\nTổng điểm thi tốt nghiệp THPT theo tổ hợp môn đăng ký xét tuyển năm 2024 đạt từ 26,00 điểm trở lên (có bao gồm điểm ưu tiên).\nCó chứng chỉ tiếng Anh IELTS 7.5 trở lên hoặc tương đương hoặc chứng chỉ ngoại ngữ tiếng Nhật N2 trở lên (với thí sinh ĐKXT vào ngành Ngôn ngữ Nhật) hoặc chứng chỉ ngoại ngữ tiếng Hàn Quốc TOPIK

In [56]:
API_URL = "https://stock.cmcts.ai/c-agent/api/v1/prediction/d8e6fd42-9a4f-4cb5-9820-62356eda3758"
# API_URL = "https://stock.cmcts.ai/c-agent/api/v1/prediction/f8626768-f118-414d-a89b-f6c7c6d5b66b"

In [57]:
def query(payload):
    response = requests.post(API_URL, json=payload)
    return response.json()

In [58]:
realAnswer_fill_values = {(index, "Real answer"): "" for index in range(df.shape[0])}
qualityScore_fill_values = {(index, "Score"): "" for index in range(df.shape[0])}
note_fill_values = {(index, "Note"): "" for index in range(df.shape[0])}

realAnswer_fill_values, qualityScore_fill_values, note_fill_values

({(0, 'Real answer'): '',
  (1, 'Real answer'): '',
  (2, 'Real answer'): '',
  (3, 'Real answer'): ''},
 {(0, 'Score'): '', (1, 'Score'): '', (2, 'Score'): '', (3, 'Score'): ''},
 {(0, 'Note'): '', (1, 'Note'): '', (2, 'Note'): '', (3, 'Note'): ''})

In [59]:
results = []
for index, (question, true_answer) in enumerate(zip(questions, true_answers)):
    response = query({
        "question": f"""Question: {question}
    True answer: {true_answer}""",
    # "chatId": "https://stock.cmcts.ai/c-agent/api/v1/prediction/84044770-5696-4600-ab12-377985460485"
    # "stream": True
    })
    results.append(response['agentReasoning'][1]['state'])

    realAnswer_fill_values[index, "Real answer"] = results[-1]["real_answer"]
    qualityScore_fill_values[index, "Score"] = results[-1]["quality_score"]
    note_fill_values[index, "Note"] = results[-1]["note"]

In [60]:
results

[{'question': 'Chính sách học bổng của trường',
  'true_answer': 'Chính sách học bổng\n1. Học bổng CMC Khai phóng\nSố lượng dự kiến: 50 suất.\n\nTrị giá học bổng: 100% học phí toàn khóa học (không bao gồm học phí chương trình học tiếng Anh).\n\nĐối tượng và tiêu chí xét học bổng: thí sinh nhập học Trường Đại học CMC năm 2024 thỏa mãn ít nhất một trong các tiêu chí sau:\n\nĐạt giải nhất, nhì, ba trong các kỳ thi học sinh giỏi và kỳ thi Khoa học Kỹ thuật cấp quốc gia, quốc tế trong các năm từ 2021 – 2024 và có môn đoạt giải nằm trong tổ hợp xét tuyển vào ngành tương ứng của Trường Đại học CMC.\nLưu ý: Thí sinh đạt giải môn Tin học được xét học bổng vào ngành Công nghệ Thông tin.\n\nTổng điểm thi tốt nghiệp THPT theo tổ hợp môn đăng ký xét tuyển năm 2024 đạt từ 26,00 điểm trở lên (có bao gồm điểm ưu tiên).\nCó chứng chỉ tiếng Anh IELTS 7.5 trở lên hoặc tương đương hoặc chứng chỉ ngoại ngữ tiếng Nhật N2 trở lên (với thí sinh ĐKXT vào ngành Ngôn ngữ Nhật) hoặc chứng chỉ ngoại ngữ tiếng Hàn 

In [61]:
realAnswer_fill_values

{(0,
  'Real answer'): '**Chính sách học bổng của Trường Đại học CMC năm 2024**\n\nTrường Đại học CMC có nhiều chính sách học bổng hấp dẫn dành cho các thí sinh có thành tích học tập xuất sắc. Dưới đây là một số thông tin chi tiết về các loại học bổng:\n\n### 1. Học bổng CMC Khai phóng\n- **Số lượng**: 50 suất\n- **Trị giá**: 100% học phí toàn khóa học (không bao gồm học phí chương trình học tiếng Anh)\n- **Tiêu chí xét học bổng**:\n  - Đạt giải nhất, nhì, ba trong các kỳ thi học sinh giỏi và kỳ thi Khoa học Kỹ thuật cấp quốc gia, quốc tế từ 2021 – 2024.\n  - Tổng điểm thi tốt nghiệp THPT theo tổ hợp môn đăng ký xét tuyển năm 2024 đạt từ 26,00 điểm trở lên.\n  - Có chứng chỉ tiếng Anh IELTS 7.5 trở lên hoặc tương đương.\n\n### 2. Học bổng CMC Sáng tạo\n- **Số lượng**: 100 suất\n- **Trị giá**: 70% học phí toàn khóa học\n- **Tiêu chí xét học bổng**:\n  - Đạt giải nhất, nhì, ba các cuộc thi học sinh giỏi cấp tỉnh, thành phố từ 2021 – 2024.\n  - Kết quả học tập THPT lớp 10, lớp 11, HK1 lớp

In [63]:
print(results[-1]["note"])

Đánh giá chi tiết về câu trả lời:

Điểm mạnh:
- Cung cấp thông tin chi tiết về các khoản học phí
- Trình bày rõ ràng, có cấu trúc
- Bao gồm thông tin về học phí chính khóa và học phí tiếng Anh

Điểm hạn chế:
- Không đề cập đến các khoản phí cụ thể được nêu trong đáp án gốc:
  1. Phí khám sức khỏe (170.000 đồng)
  2. Bảo hiểm y tế 12 tháng (884.520 đồng)
  3. Sinh hoạt phí Giáo dục Quốc phòng và An ninh (1.800.000 đồng)

Sai khác quan trọng:
- Bỏ qua thông tin về học phí tạm thu ban đầu (11.054.520 đồng)
- Không nhắc đến hướng dẫn tra cứu giấy báo nhập học qua mã QR
- Thiếu chi tiết về chính sách chuyển đổi học phí cho các trường hợp miễn học tiếng Anh

Đề xuất cải thiện:
- Bổ sung đầy đủ các khoản phí theo đáp án gốc
- Thêm thông tin về quy trình chuyển đổi học phí
- Đưa ra link tra cứu chính thức của trường


In [16]:
cmc_uni_response = query({"question": "Chính sách học bổng"})

In [17]:
cmc_uni_response

{'text': '**Chính sách học bổng và ưu đãi năm 2024 của Trường Đại học CMC**\n\nTrường Đại học CMC đã công bố chính sách học bổng và ưu đãi cho năm 2024 với tổng trị giá 96 tỷ đồng nhằm hỗ trợ các thí sinh có thành tích học tập xuất sắc. Dưới đây là các loại học bổng và điều kiện xét tuyển:\n\n### 1. Học bổng CMC Khai phóng\n- **Số lượng**: 50 suất\n- **Trị giá**: 100% học phí toàn khóa học (không bao gồm học phí chương trình học tiếng Anh)\n- **Điều kiện**:\n  - Đạt giải nhất, nhì, ba trong các kỳ thi học sinh giỏi và kỳ thi Khoa học Kỹ thuật cấp quốc gia, quốc tế từ 2021 – 2024.\n  - Tổng điểm thi tốt nghiệp THPT theo tổ hợp môn đăng ký xét tuyển năm 2024 đạt từ 26,00 điểm trở lên.\n  - Có chứng chỉ tiếng Anh IELTS 7.5 trở lên hoặc tương đương.\n\n### 2. Học bổng CMC Sáng tạo\n- **Số lượng**: 100 suất\n- **Trị giá**: 70% học phí toàn khóa học\n- **Điều kiện**:\n  - Đạt giải nhất, nhì, ba các cuộc thi học sinh giỏi cấp tỉnh, thành phố từ 2021 – 2024.\n  - Kết quả học tập THPT lớp 10, l

{'question': 'Chính sách học bổng của trường',
 'true_answer': 'Chính sách học bổng\n1. Học bổng CMC Khai phóng\nSố lượng dự kiến: 50 suất.\n\nTrị giá học bổng: 100% học phí toàn khóa học (không bao gồm học phí chương trình học tiếng Anh).\n\nĐối tượng và tiêu chí xét học bổng: thí sinh nhập học Trường Đại học CMC năm 2024 thỏa mãn ít nhất một trong các tiêu chí sau:\n\nĐạt giải nhất, nhì, ba trong các kỳ thi học sinh giỏi và kỳ thi Khoa học Kỹ thuật cấp quốc gia, quốc tế trong các năm từ 2021 – 2024 và có môn đoạt giải nằm trong tổ hợp xét tuyển vào ngành tương ứng của Trường Đại học CMC.\nLưu ý: Thí sinh đạt giải môn Tin học được xét học bổng vào ngành Công nghệ Thông tin.\n\nTổng điểm thi tốt nghiệp THPT theo tổ hợp môn đăng ký xét tuyển năm 2024 đạt từ 26,00 điểm trở lên (có bao gồm điểm ưu tiên).\nCó chứng chỉ tiếng Anh IELTS 7.5 trở lên hoặc tương đương hoặc chứng chỉ ngoại ngữ tiếng Nhật N2 trở lên (với thí sinh ĐKXT vào ngành Ngôn ngữ Nhật) hoặc chứng chỉ ngoại ngữ tiếng Hàn Qu

In [20]:
for sourceDoc in cmc_uni_response['agentReasoning'][1]['sourceDocuments']:
    print(sourceDoc['metadata']['source'])    

s3://cts-llm-docs-bucket/cmcuni-sample-ver-2/du-lieu-tuyen-sinh-web/hoc-phi-hoc-bong/chinh-sach-hoc-bong.txt
s3://cts-llm-docs-bucket/cmcuni-sample-ver-2/du-lieu-tuyen-sinh-2024/thong-tin-tuyen-sinh/Chính sách Học bổng, ưu đãi năm 2024.pdf
s3://cts-llm-docs-bucket/cmcuni-sample-ver-2/du-lieu-tuyen-sinh-2024/thong-tin-tuyen-sinh/Hướng dẫn đăng ký xét tuyển học bạ trực tuyến vào Trường Đại học CMC năm 2024.pdf
s3://cts-llm-docs-bucket/cmcuni-sample-ver-2/du-lieu-tuyen-sinh-web/xet-tuyen-tuyen-sinh/huong-dan-dang-ky-xet-tuyen-hoc-ba-truc-tuyen-vao-truong-dai-hoc-cmc-nam-2024.txt
s3://cts-llm-docs-bucket/cmcuni-sample-ver-2/du-lieu-tuyen-sinh-web/hoc-phi-hoc-bong/chinh-sach-hoc-bong.txt
s3://cts-llm-docs-bucket/cmcuni-sample-ver-2/du-lieu-tuyen-sinh-web/hoc-phi-hoc-bong/chinh-sach-hoc-bong.txt
s3://cts-llm-docs-bucket/cmcuni-sample-ver-2/du-lieu-tuyen-sinh-web/hoc-phi-hoc-bong/chinh-sach-hoc-bong.txt
s3://cts-llm-docs-bucket/cmcuni-sample-ver-2/du-lieu-tuyen-sinh-web/hoc-phi-hoc-bong/chinh

In [19]:
cmc_uni_response['agentReasoning'][1]['sourceDocuments']


[{'pageContent': 'Chính sách Học bổng, ưu đãi năm 2024 20/02/2022 2024-04-16 8:04 Chính sách Học bổng, ưu đãi năm 2024 Chính sách Học bổng, ưu đãi năm 2024 Tập đoàn Công nghệ CMC dành quỹ học bổng, ưu đãi “ CMC – Vì bạn xứng đáng ” trị giá 96 tỷ đồng cho những thí sinh có thành tích học tập xuất sắc nhập học năm 2024 nhằm phát triển nguồn nhân lực chất lượng cao tại Việt Nam. 1. Chính sách học bổng 1.1 Học bổng CMC Khai phóng Số lượng dự kiến: 50 suất. Trị giá học bổng: 100% học phí toàn khóa học (không bao gồm học phí chương trình học tiếng Anh). Đối tượng và tiêu chí xét học bổng: thí sinh nhập học Trường Đại học CMC năm 2024 thỏa mãn ít nhất một trong các tiêu chí sau: Đạt giải nhất, nhì, ba trong các kỳ thi học sinh giỏi và kỳ thi Khoa học Kỹ thuật cấp quốc gia, quốc tế trong các năm từ 2021 – 2024 và có môn đoạt giải nằm trong tổ hợp xét tuyển vào ngành tương ứng của Trường Đại học CMC. Lưu ý : Thí sinh đạt giải môn Tin học được xét học bổng vào ngành Công nghệ Thông tin.',
  'met